# Spherical Pendulum on S²

Derives the equations of motion for a particle constrained to move on a sphere,
using the global geometric formulation from Lee, Leok, McClamroch (2018), Chapter 5.

**Configuration manifold**: S² = {q ∈ ℝ³ : ‖q‖ = 1}

**Kinematics**: q̇ = ω × q, where ω ∈ ℝ³ is the angular velocity with ω ⊥ q

**EOM** (Lee et al. Eq 5.14): ml²ω̇ + mgl(q × e₃) = f

In [1]:
# Install from GitHub (for Colab); uncomment if not installed locally
# !pip install -q git+https://github.com/vkotaru/pygeomech.git@geomech

from geomech import (
    S2, Scalar, Vector, Dot, Cross,
    SystemVariables, TimeDerivative,
    compute_eom, to_standard_form, getScalars,
    to_latex, display_latex, display_eom, display_standard_form,
)
from geomech.utils.printing import print_tree
from IPython.display import Math

## 1. Define the system

In [2]:
# Parameters
m, g, l = getScalars('m g l', attr=['Constant'])
e3 = Vector('e3', attr=['Constant'])

# Configuration variable on S²
q = S2('q')
omega = q.get_tangent_vector()   # angular velocity ω ∈ T_q S²
xi = q.get_variation_vector()     # variation vector ξ ∈ T_q S²

# External force in tangent space
f = Vector('f')

print('q:', q, '  (S2 manifold point)')
print('ω:', omega, '  (tangent vector)')
print('ξ:', xi, '  (variation vector)')

q: q   (S2 manifold point)
ω: \omega_{q}   (tangent vector)
ξ: \xi_{q}   (variation vector)


## 2. Lagrangian

$$L = \frac{1}{2} m l^2 \omega \cdot \omega - m g l \, q \cdot e_3$$

In [3]:
half = Scalar('0.5', value=0.5, attr=['Constant'])

KE = half * m * l * l * Dot(omega, omega)
PE = m * g * l * Dot(q, e3)
L = KE - PE

display(Math(r'KE = ' + to_latex(KE)))
display(Math(r'PE = ' + to_latex(PE)))
display(Math(r'L = ' + to_latex(L)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 3. Infinitesimal work

$$\delta W = \xi \cdot f$$

where f is a force in the tangent space T_q S².

In [4]:
dW = Dot(xi, f)
display(Math(r'\delta W = ' + to_latex(dW)))

<IPython.core.display.Math object>

## 4. Derive equations of motion

The pipeline:
1. Compute δL (variation of Lagrangian)
2. Form δS = δL + δW
3. Apply manifold rules: δ(ω) = ξ̇ − ω × ξ
4. Simplify (vector identities, BAC-CAB)
5. Integration by parts (move d/dt off ξ)
6. Expand and extract coefficient of ξ

In [5]:
variables = SystemVariables(vectors=[q])
eom = compute_eom(L, dW, variables)

key = list(eom.keys())[0]
_, eqn = eom[key]

print('EOM:')
display_eom(eom)

EOM:


<IPython.core.display.Math object>

## 5. Expression tree

In [6]:
print_tree(eqn)

                                         VAdd
              /                                    \                     \
             S*V                                  S*V                   v:f
      /               \                  /                    \
    d/dt             Mul               Cross                 Mul
      |         /           \         /      \            /        \
v:\omega_{q}   -1*         Mul      S2:q   v:e3*         Mul      -1*
                        /       \                     /       \
                       Mul     l*                    Mul     l*
                      /    \                        /    \
                     m*   l*                       m*   g*

expr: (d/dt(\omega_{q})*-m*l*l + cross(q, e3)*-m*g*l + f)


## 6. Standard form: M(q)·ω̇ + f(q,ω) + G(q)·u = 0

In [7]:
sf = to_standard_form(eom, variables, [f])
display_standard_form(sf)

<IPython.core.display.Math object>